In [7]:
# %%
%pip install scikit-learn pandas numpy optuna xgboost lightgbm catboost


[notice] A new release of pip is available: 26.0 -> 26.0.1
[notice] To update, run: python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [8]:
# %%
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

TRAIN_DATA = pd.read_csv('train-data.csv', index_col='id')
TRAIN_LABEL = pd.read_csv('train-label.csv', index_col='id')
TEST_DATA = pd.read_csv('test-data.csv', index_col='id')

if 'subscription' in TRAIN_DATA.columns:
    TRAIN_DATA = TRAIN_DATA.drop(columns=['subscription'])

def add_features(df):
    df = df.copy()
    df['contacted_recently'] = ((df['pdays'] != -1) & (df['pdays'] < 30)).astype(int)
    df['prev_success'] = (df['poutcome'] == 'SUC').astype(int)
    df['never_contacted'] = (df['previous'] == 0).astype(int)
    df['pdays_recent'] = df['pdays'].apply(lambda x: 0 if x == -1 else x)
    df['multiple_prev_contacts'] = (df['previous'] > 2).astype(int)
    df['very_short_call'] = (df['duration'] < 30).astype(int)
    df['short_call'] = (df['duration'] < 60).astype(int)
    df['medium_call'] = ((df['duration'] >= 60) & (df['duration'] <= 300)).astype(int)
    df['long_call'] = (df['duration'] > 300).astype(int)
    df['very_long_call'] = (df['duration'] > 600).astype(int)
    df['duration_bucket'] = pd.cut(
        df['duration'], bins=[-1, 30, 60, 180, 300, 600, 99999], labels=[0, 1, 2, 3, 4, 5]
    ).astype(int)
    df['debt'] = (df['balance'] < 0).astype(int)
    df['has_balance'] = (df['balance'] > 0).astype(int)
    df['medium_balance'] = ((df['balance'] > 0) & (df['balance'] <= 1000)).astype(int)
    df['high_balance'] = (df['balance'] > 1000).astype(int)
    df['very_high_balance'] = (df['balance'] > 5000).astype(int)
    df['log_balance'] = np.log1p(df['balance'].clip(lower=0))
    df['first_contact'] = (df['campaign'] == 1).astype(int)
    df['over_contacted'] = (df['campaign'] > 5).astype(int)
    df['log_campaign'] = np.log1p(df['campaign'])
    df['is_young'] = (df['age'] < 30).astype(int)
    df['is_middle_age'] = ((df['age'] >= 30) & (df['age'] <= 60)).astype(int)
    df['is_retired_age'] = (df['age'] > 60).astype(int)
    df['long_call_prev_success'] = df['long_call'] * df['prev_success']
    df['long_call_never_contacted'] = df['long_call'] * df['never_contacted']
    df['high_balance_long_call'] = df['high_balance'] * df['long_call']
    df['success_signal'] = ((df['duration'] > 300) & (df['poutcome'] == 'SUC')).astype(int)
    df['warm_lead'] = ((df['contacted_recently'] == 1) & (df['prev_success'] == 1)).astype(int)
    df['cold_lead'] = ((df['never_contacted'] == 1) & (df['short_call'] == 1)).astype(int)
    df['q1'] = df['month'].isin([1, 2, 3]).astype(int)
    df['q2'] = df['month'].isin([4, 5, 6]).astype(int)
    df['q3'] = df['month'].isin([7, 8, 9]).astype(int)
    df['q4'] = df['month'].isin([10, 11, 12]).astype(int)
    return df

TRAIN_DATA = add_features(TRAIN_DATA)
TEST_DATA  = add_features(TEST_DATA)

print('TRAIN_DATA shape:', TRAIN_DATA.shape)
print('TEST_DATA shape: ', TEST_DATA.shape)

TRAIN_DATA shape: (29839, 49)
TEST_DATA shape:  (19893, 49)


In [9]:
# %%
from sklearn.preprocessing import OrdinalEncoder
from sklearn.model_selection import StratifiedKFold

cat_cols = ['job', 'marital_status', 'education', 'default_loan',
            'housing_loan', 'personal_loan', 'contact_type', 'poutcome']

num_cols = [c for c in TRAIN_DATA.columns if c not in cat_cols]

ENCODER = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1).fit(TRAIN_DATA[cat_cols])

def preprocess(df, encoder):
    df = df.copy()
    cat_enc = pd.DataFrame(encoder.transform(df[cat_cols]), columns=cat_cols, index=df.index)
    num_df = df[num_cols].copy()
    return pd.concat([cat_enc, num_df], axis=1)

X_train = preprocess(TRAIN_DATA, ENCODER)
X_test  = preprocess(TEST_DATA,  ENCODER)
y_train = TRAIN_LABEL['subscription'].values

def target_encode_cv(X_tr, y_tr, X_te, cols, n_splits=5):
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
    X_tr_enc = X_tr.copy()
    X_te_enc  = X_te.copy()
    global_mean = y_tr.mean()

    for col in cols:
        oof = np.full(len(X_tr), global_mean)
        te_vals = np.zeros(len(X_te))

        for fold_tr_idx, fold_val_idx in skf.split(X_tr, y_tr):
            means = y_tr.iloc[fold_tr_idx].groupby(X_tr[col].iloc[fold_tr_idx]).mean()
            oof[fold_val_idx] = X_tr[col].iloc[fold_val_idx].map(means).fillna(global_mean).values
            te_vals += X_te[col].reset_index(drop=True).map(means).fillna(global_mean).values / n_splits

        X_tr_enc[col + '_te'] = oof
        X_te_enc[col  + '_te'] = te_vals

    return X_tr_enc, X_te_enc

y_series = pd.Series(y_train, index=X_train.index)
X_train_te, X_test_te = target_encode_cv(X_train, y_series, X_test, cat_cols)

print('X_train_te shape:', X_train_te.shape)
print('X_test_te  shape:', X_test_te.shape)

X_train_te shape: (29839, 57)
X_test_te  shape: (19893, 57)


In [10]:
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import balanced_accuracy_score
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.ensemble import HistGradientBoostingClassifier
from catboost import CatBoostClassifier

N_SPLITS = 10
skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=42)
scale_pos = (y_train == 0).sum() / (y_train == 1).sum()

X_arr    = X_train_te.values
X_te_arr = X_test_te.values

# Our Optuna-tuned params from previous runs
hgbm_params = {
    'learning_rate': 0.016782184919286597,
    'max_iter': 1117,
    'max_leaf_nodes': 26,
    'max_depth': 5,
    'min_samples_leaf': 72,
    'l2_regularization': 2.2767559805786015,
}

xgb_params = {
    'n_estimators':     899,
    'learning_rate':    0.044925311663262746,
    'max_depth':        4,
    'min_child_weight': 49,
    'subsample':        0.9066264844754309,
    'colsample_bytree': 0.9404726184518012,
    'reg_alpha':        0.5796743517191622,
    'reg_lambda':       2.9719182594187536,
    'gamma':            1.0300044484691284,
    'scale_pos_weight': scale_pos,
    'eval_metric':      'logloss',
    'random_state':     42,
    'n_jobs':           -1,
}

lgbm_params = {
    'n_estimators':      654,
    'learning_rate':     0.025695396965748477,
    'max_depth':         7,
    'num_leaves':        24,
    'min_child_samples': 23,
    'subsample':         0.8156760979769445,
    'colsample_bytree':  0.7770742352579232,
    'reg_alpha':         2.8693357936064383,
    'reg_lambda':        4.335019813134123,
    'class_weight':      'balanced',
    'random_state':      42,
    'n_jobs':            -1,
    'verbose':           -1,
}

# CatBoost — tuned conservatively for balanced accuracy
cat_params = {
    'iterations':         1200,
    'learning_rate':      0.02,
    'depth':              6,
    'l2_leaf_reg':        3.0,
    'auto_class_weights': 'Balanced',
    'eval_metric':        'Logloss',
    'random_seed':        42,
    'verbose':            0,
}

# OOF and test arrays
model_names = ['HGBM', 'XGB', 'LGBM', 'CAT']
oof_preds  = {name: np.zeros(len(y_train))   for name in model_names}
test_preds = {name: np.zeros(len(X_test_te)) for name in model_names}

for fold, (tr_idx, val_idx) in enumerate(skf.split(X_arr, y_train)):
    X_tr, X_val = X_arr[tr_idx], X_arr[val_idx]
    y_tr, y_val = y_train[tr_idx], y_train[val_idx]

    m_hgbm = HistGradientBoostingClassifier(
        class_weight='balanced', random_state=42,
        early_stopping=False, **hgbm_params)
    m_hgbm.fit(X_tr, y_tr)
    oof_preds['HGBM'][val_idx]  = m_hgbm.predict_proba(X_val)[:, 1]
    test_preds['HGBM']         += m_hgbm.predict_proba(X_te_arr)[:, 1] / N_SPLITS

    m_xgb = XGBClassifier(**xgb_params)
    m_xgb.fit(X_tr, y_tr)
    oof_preds['XGB'][val_idx]   = m_xgb.predict_proba(X_val)[:, 1]
    test_preds['XGB']          += m_xgb.predict_proba(X_te_arr)[:, 1] / N_SPLITS

    m_lgbm = LGBMClassifier(**lgbm_params)
    m_lgbm.fit(X_tr, y_tr)
    oof_preds['LGBM'][val_idx]  = m_lgbm.predict_proba(X_val)[:, 1]
    test_preds['LGBM']         += m_lgbm.predict_proba(X_te_arr)[:, 1] / N_SPLITS

    m_cat = CatBoostClassifier(**cat_params)
    m_cat.fit(X_tr, y_tr)
    oof_preds['CAT'][val_idx]   = m_cat.predict_proba(X_val)[:, 1]
    test_preds['CAT']          += m_cat.predict_proba(X_te_arr)[:, 1] / N_SPLITS

    print(f'Fold {fold+1}/{N_SPLITS} done')

print('\nOOF Balanced Accuracy per model:')
for name in model_names:
    best_ba, best_t = 0.0, 0.5
    for t in np.arange(0.1, 0.9, 0.005):
        ba = balanced_accuracy_score(y_train, (oof_preds[name] >= t).astype(int))
        if ba > best_ba:
            best_ba, best_t = ba, t
    print(f'  {name}: {best_ba:.4f}  (best t={best_t:.3f})')

Fold 1/10 done
Fold 2/10 done
Fold 3/10 done
Fold 4/10 done
Fold 5/10 done
Fold 6/10 done
Fold 7/10 done
Fold 8/10 done
Fold 9/10 done
Fold 10/10 done

OOF Balanced Accuracy per model:
  HGBM: 0.8736  (best t=0.415)
  XGB: 0.8709  (best t=0.440)
  LGBM: 0.8722  (best t=0.405)
  CAT: 0.8738  (best t=0.430)


In [11]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold

# ── Step 1: Stacking meta-model with CV to prevent leakage ───────────────────
# We use a nested CV — meta-model is trained on held-out OOF predictions only
meta_skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=99)

oof_stack  = np.zeros(len(y_train))   # final stacked OOF probs
oof_matrix = np.column_stack([oof_preds[n] for n in model_names])
test_matrix = np.column_stack([test_preds[n] for n in model_names])

for fold, (tr_idx, val_idx) in enumerate(meta_skf.split(oof_matrix, y_train)):
    meta = LogisticRegression(
        class_weight='balanced', max_iter=1000, random_state=42, C=0.1
    )
    meta.fit(oof_matrix[tr_idx], y_train[tr_idx])
    oof_stack[val_idx] = meta.predict_proba(oof_matrix[val_idx])[:, 1]

# ── Step 2: Train final meta-model on all OOF for test predictions ────────────
final_meta = LogisticRegression(
    class_weight='balanced', max_iter=1000, random_state=42, C=0.1
)
final_meta.fit(oof_matrix, y_train)
test_stack = final_meta.predict_proba(test_matrix)[:, 1]

print('Meta-model coefficients (HGBM, XGB, LGBM, CAT):')
print([f'{c:.3f}' for c in final_meta.coef_[0]])

# ── Step 3: Find best threshold on stacked OOF ───────────────────────────────
best_ba, best_threshold = 0.0, 0.5

for t in np.arange(0.1, 0.9, 0.001):
    preds = (oof_stack >= t).astype(int)
    ba    = balanced_accuracy_score(y_train, preds)
    if ba > best_ba:
        best_ba        = ba
        best_threshold = t

print(f'\nStacked OOF BA:  {best_ba:.5f}')
print(f'Best threshold:  {best_threshold:.3f}')

# ── Step 4: Show threshold table around the best ──────────────────────────────
print('\nThreshold | Pred dist 1 | OOF BA')
print('-' * 40)
for t in np.arange(best_threshold - 0.05, best_threshold + 0.05, 0.005):
    preds = (oof_stack >= t).astype(int)
    ba    = balanced_accuracy_score(y_train, preds)
    marker = ' ← best' if abs(t - best_threshold) < 0.001 else ''
    print(f'  {t:.3f}   |    {preds.sum()}      | {ba:.4f}{marker}')

Meta-model coefficients (HGBM, XGB, LGBM, CAT):
['1.193', '1.153', '1.548', '2.487']

Stacked OOF BA:  0.87483
Best threshold:  0.453

Threshold | Pred dist 1 | OOF BA
----------------------------------------
  0.403   |    7461      | 0.8739
  0.408   |    7420      | 0.8740
  0.413   |    7379      | 0.8738
  0.418   |    7336      | 0.8742
  0.423   |    7307      | 0.8742
  0.428   |    7272      | 0.8744
  0.433   |    7244      | 0.8743
  0.438   |    7217      | 0.8740
  0.443   |    7187      | 0.8742
  0.448   |    7154      | 0.8744
  0.453   |    7113      | 0.8748 ← best
  0.458   |    7078      | 0.8742
  0.463   |    7046      | 0.8743
  0.468   |    7015      | 0.8743
  0.473   |    6988      | 0.8738
  0.478   |    6959      | 0.8735
  0.483   |    6925      | 0.8735
  0.488   |    6890      | 0.8732
  0.493   |    6862      | 0.8729
  0.498   |    6828      | 0.8729
  0.503   |    6789      | 0.8727


In [12]:
# Apply threshold to stacked test probabilities
test_classes = (test_stack >= best_threshold).astype(int)

print(f'Prediction distribution — 0: {(test_classes==0).sum()}, 1: {(test_classes==1).sum()}')
print(f'Stacked OOF BA: {best_ba:.5f}')
print(f'Threshold used: {best_threshold:.3f}')

Prediction distribution — 0: 15162, 1: 4731
Stacked OOF BA: 0.87483
Threshold used: 0.453


In [14]:
submission = pd.DataFrame({
    'id':           TEST_DATA.index,
    'subscription': test_classes
})
submission.to_csv('submissiongen.csv', index=False)
print('Saved! Preview:')
print(submission.head())

Saved! Preview:
      id  subscription
0  37797             0
1  37798             0
2  37799             0
3  37800             0
4  37801             1
